# SetUp

In [1]:
import mlflow
import os

import matplotlib as mpl
import matplotlib.pyplot as plt
import json
from tqdm import tqdm

In [2]:
def legend(ax1, ax2=None, extra_lines=[], extra_labels=[], loc=None):
    lines, labels = ax1.get_legend_handles_labels()
    if ax2 is not None:
        lines2, labels2 = ax2.get_legend_handles_labels()
    else:
        lines2, labels2 = [], []
    by_label = dict(zip(labels + labels2 + extra_labels, lines + lines2 + extra_lines))
    return plt.legend(by_label.values(), by_label.keys(), loc=loc)
    # plt.legend(lines + lines2, labels + labels2)


colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

In [3]:
%cd '/home/tau/sdouka/codebase/experimental_grow'
os.getcwd()

/home/tau/sdouka/codebase/experimental_grow


'/home/tau/sdouka/codebase/experimental_grow'

# Get Experiments

In [4]:
client = mlflow.client.MlflowClient()

/home/tau/sdouka/miniconda3/envs/gromo/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:140: FutureWarning: Filesystem tracking backend (e.g., './mlruns') is deprecated. Please switch to a database backend (e.g., 'sqlite:///mlflow.db'). For feedback, see: https://github.com/mlflow/mlflow/issues/18534
  return FileStore(store_uri, store_uri)


In [5]:
def get_runs(exp_name):
    experiment = client.get_experiment_by_name(exp_name)
    runs = client.search_runs([experiment.experiment_id])
    print("Number of runs", len(runs))
    runIds = {run.info.run_name: run.info.run_id for run in runs}
    return runIds

In [6]:
def get_logs(runIDs):
    if not isinstance(runIDs, list):
        runIDs = [runIDs]
    print(runIDs)

    run_info = {}
    run_info_t = {}
    for runID in runIDs:
        # run_info["steps"].append({item.step: item.step for item in client.get_metric_history})
        retreived_run = client.get_run(runID)

        # Timing
        start_time = retreived_run.info.start_time
        try:
            run_info["duration (ms)"] = retreived_run.info.end_time - start_time
            run_info_t["duration (ms)"] = retreived_run.info.end_time - start_time
            run_info["duration (s)"] = run_info["duration (ms)"] / 1_000
            run_info_t["duration (s)"] = run_info_t["duration (ms)"] / 1_000
            run_info["duration (h)"] = run_info["duration (s)"] / 3_600
            run_info_t["duration (h)"] = run_info_t["duration (s)"] / 3_600
        except:
            print("Could not get duration of run")

        # Run name
        run_name = retreived_run.info.run_name
        if "name" not in run_info:
            run_info["name"] = []
            run_info_t["name"] = []
        run_info["name"].append(run_name)
        run_info_t["name"].append(run_name)

        # Metrics
        for key in tqdm(retreived_run.data.metrics.keys()):
            if ("/node " in key) or ("/edge " in key) or ("/layer " in key):
                continue
            if (
                ("_utilization_" in key)
                or ("_percentage" in key)
                or ("network_" in key)
            ):
                continue

            if key not in run_info:
                run_info[key] = []
                run_info_t[key] = []
            metric_data = client.get_metric_history(runID, key)
            run_info[key].append({item.step: item.value for item in metric_data})
            run_info_t[key].append(
                {
                    (item.timestamp - start_time) / 1_000: item.value
                    for item in metric_data
                }
            )

        # Artifacts
        try:
            # mlflow.artifacts.load_dict("mlflow-artifacts:/235085688558347327/e95c2ea3c22f44cb950829023dffc42e/artifacts/gh.json")
            artifact_uri = mlflow.artifacts.download_artifacts(
                run_id=runID, artifact_path="gh.json"
            )
            with open(artifact_uri, "r") as f:
                data = json.load(f)
            print(f"Got growth history for {artifact_uri}")
            if "growth history" not in run_info:
                run_info["growth history"] = []
            run_info["growth history"].append(data)
        except mlflow.MlflowException as err:
            print(f"[Run {runID}] {err}")
        except Exception as err:
            pass

    return run_info, run_info_t

In [7]:
runIds = get_runs("Strategies")
runIds

Number of runs 9


{'funny-smelt-612': '33491a88493146328cbd077a9d612a3c',
 'wistful-pig-358': '79f54e85b2474707b37bd180bef5d959',
 'sincere-goat-473': '2e1b912fc3bc44e5a18eaf23853e701b',
 'spiffy-smelt-549': '68d5fc3570554d7785a4c3806ef2c083',
 'epochs200': '8488aa037ef24eefa3cd8de21bbbde60',
 'bustling-mink-443': 'f9a735e52e974901983c9eabf404bdca',
 'mysterious-shoat-45': '4d7ba598fa9543949500e9c2952f733d',
 'defiant-lark-710': 'caaa4e30d1ca47f7b39fb299ca61e16a',
 'amusing-foal-630': '9e66bc1d2f5e4a82a7856f54f3bf6611'}

In [ ]:
run_info, run_info_t = get_logs([runIds["amusing-foal-630"], runIds["sincere-goat-473"]])
list(run_info.keys())

['9e66bc1d2f5e4a82a7856f54f3bf6611', '2e1b912fc3bc44e5a18eaf23853e701b']


100%|██████████| 104/104 [00:00<00:00, 335.50it/s]


# Plots

In [ ]:
crop = None
fig, ax = plt.subplots(figsize=(12, 5))
for i in range(len(run_info["name"])):
    ax.plot(
        list(run_info["training.train accuracy"][i].keys())[:crop],
        list(run_info["training.train accuracy"][i].values())[:crop],
        label=f'{run_info["name"][i]} train',
    )
    ax.plot(
        list(run_info["training.test accuracy"][i].keys())[:crop],
        list(run_info["training.test accuracy"][i].values())[:crop],
        label=f'{run_info["name"][i]} test',
    )
plt.ylabel("accuracy")
plt.xlabel("epochs")
legend(ax)
plt.show()